# apertus-eval-prep — paper-matrix **vLLM backend** (Colab ID-5)

[Open in Colab](https://colab.research.google.com/github/Shivani767/apertus-eval-prep/blob/master/notebooks/colab_stability_backend.ipynb)

Runtime → **T4 GPU**.

Do **not** Run all. Every session: **cell 1 → cell 2 → cell 3 (Drive) → one sweep cell**.

**Install rule:** never bare `pip install vllm` and never `pip install -e ".[gpu]"` on Colab — PyPI vLLM is CUDA-13 and breaks with `libcudart.so.13`. Cell 2 installs the official **`+cu129`** GitHub wheel and keeps Colab's existing `torch`.

Drive folder (shared with HF runs): `MyDrive/apertus-eval-prep-paper`.


In [ ]:
# Cell 1 — clone or pull (no vLLM here)
import os
if os.path.exists("pyproject.toml") and os.path.exists("src/apertus_eval_prep"):
    print("Already in repo root")
    !git pull --ff-only
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
    !git pull --ff-only
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
# [viz] only — [gpu] pulls a broken PyPI vllm wheel.
!pip -q install -e ".[viz]"
!git log -1 --oneline


In [ ]:
# Cell 2 — GPU check + install vLLM cu129 wheel (rerun every session)
import os, sys, subprocess
from pathlib import Path

if Path("pyproject.toml").exists() and Path("src/apertus_eval_prep").exists():
    pass
elif Path("apertus-eval-prep/pyproject.toml").exists():
    os.chdir("apertus-eval-prep")
else:
    raise FileNotFoundError("Run cell 1 first.")

!git pull --ff-only
!pip -q install -e ".[viz]"
!git log -1 --oneline

_repo_src = str((Path.cwd() / "src").resolve())
if _repo_src not in sys.path:
    sys.path.insert(0, _repo_src)

import torch
assert torch.cuda.is_available(), "Set runtime to GPU (T4) and rerun."
TORCH_PIN = torch.__version__
print(torch.cuda.get_device_name(0), "torch", TORCH_PIN, "cuda", torch.version.cuda)

VLLM_VER = os.environ.get("APERTUS_VLLM_PIN", "0.27.1")
VLLM_WHEEL = (
    f"https://github.com/vllm-project/vllm/releases/download/v{VLLM_VER}/"
    f"vllm-{VLLM_VER}+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
)

def _pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip", *args]
    print("+", " ".join(cmd), flush=True)
    p = subprocess.run(cmd, text=True, capture_output=True)
    if p.stdout.strip():
        print(p.stdout[-1500:], flush=True)
    if p.returncode != 0:
        print(p.stderr[-2000:], flush=True)
        if check:
            raise RuntimeError(f"pip failed ({p.returncode})")
    return p.returncode == 0

def install_vllm_colab():
    """Colab T4 (torch cu128): use the official +cu129 wheel, not PyPI cu13."""
    _pip("uninstall", "-y", "vllm", check=False)
    ok = _pip(
        "install", "-q", VLLM_WHEEL,
        "--extra-index-url", "https://download.pytorch.org/whl/cu128",
        check=False,
    )
    if not ok:
        raise RuntimeError(
            f"Could not install {VLLM_WHEEL}. "
            "Check APERTUS_VLLM_PIN or Colab Python version."
        )
    import torch as _t
    if _t.__version__ != TORCH_PIN:
        print("restoring Colab torch", TORCH_PIN, flush=True)
        _pip("install", "-q", f"torch=={TORCH_PIN}")
    for name in list(sys.modules):
        if name == "vllm" or name.startswith("vllm."):
            del sys.modules[name]
    from vllm import LLM, SamplingParams  # noqa: F401
    import vllm
    return vllm

vllm = install_vllm_colab()
print("vllm", getattr(vllm, "__version__", "?"), "wheel", VLLM_WHEEL.split("/")[-1])

if not Path("data/official/eval_set.jsonl").exists():
    !pip -q install -e ".[snapshot]"
    !python scripts/snapshot_benchmarks.py
else:
    print("official slices already on disk")


In [ ]:
# Cell 3 — Drive restore + sweep helper
import os, sys
from pathlib import Path
from google.colab import drive, files

def ensure_repo():
    for p in (Path.cwd(), Path("/content/apertus-eval-prep"), Path("apertus-eval-prep")):
        if (p / "pyproject.toml").exists() and (p / "src" / "apertus_eval_prep").exists():
            os.chdir(p)
            src = str((p / "src").resolve())
            if src not in sys.path:
                sys.path.insert(0, src)
            print("cwd:", Path.cwd(), flush=True)
            return p
    raise FileNotFoundError("Repo not found. Run cells 1–2 first.")

ensure_repo()
try:
    import apertus_eval_prep  # noqa: F401
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz]"])

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/apertus-eval-prep-paper")
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / "runs").mkdir(exist_ok=True)
Path("results/runs").mkdir(parents=True, exist_ok=True)
os.environ["APERTUS_CHECKPOINT_DIR"] = str(DRIVE / "runs")

if (DRIVE / "registry_paper.jsonl").exists():
    !cp -a {DRIVE}/registry_paper.jsonl results/registry_paper.jsonl
    print("restored registry from Drive")
if any((DRIVE / "runs").iterdir()):
    !cp -a {DRIVE}/runs/. results/runs/
    print("restored runs from Drive")
n_partial = len(list(Path("results/runs").glob("*.partial.jsonl")))
print(f"partial checkpoints on disk: {n_partial}", flush=True)

def save_paper():
    import subprocess
    DRIVE.mkdir(parents=True, exist_ok=True)
    (DRIVE / "runs").mkdir(exist_ok=True)
    if Path("results/registry_paper.jsonl").exists():
        subprocess.check_call(["cp", "-a", "results/registry_paper.jsonl", str(DRIVE / "registry_paper.jsonl")])
    if Path("results/runs").exists():
        subprocess.check_call(["bash", "-lc", f"cp -a results/runs/. {DRIVE}/runs/"])
    n = len(list(Path("results/runs").glob("*.json")))
    print(f"saved to Drive ({n} run JSON files):", DRIVE, flush=True)
    subprocess.check_call(["zip", "-r", "/tmp/paper_matrix_partial.zip", "results/runs", "results/registry_paper.jsonl"])
    files.download("/tmp/paper_matrix_partial.zip")

def sweep(*extra):
    ensure_repo()
    only_model = only_factor = None
    args = list(extra)
    i = 0
    while i < len(args):
        if args[i] == "--only-model" and i + 1 < len(args):
            only_model = args[i + 1]
            i += 2
        elif args[i] == "--only-factor" and i + 1 < len(args):
            only_factor = args[i + 1]
            i += 2
        else:
            raise ValueError(f"unknown sweep arg {args[i]!r}")
    from vllm import LLM  # noqa: F401 — fail fast if cell 2 was skipped
    from apertus_eval_prep.sweep import execute_sweep
    print(f"sweep in-process model={only_model} factor={only_factor}", flush=True)
    planned = execute_sweep(
        study_path=Path("configs/experiments/stability.yaml"),
        repo_root=Path(".").resolve(),
        out_dir=Path("results/runs"),
        registry_path=Path("results/registry_paper.jsonl"),
        profile="t4",
        only_model=only_model,
        only_factor=only_factor,
    )
    n_skip = sum(1 for p in planned if p["skipped"])
    print({"n_cells": len(planned), "n_skip": n_skip}, flush=True)
    save_paper()

!python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --dry-run --only-factor backend --out-dir results/runs --registry results/registry_paper.jsonl | head -n 20


## ID-5 sweeps — one model per session

T4 skips Qwen-7B vLLM. Run **one** cell below per Colab window.


In [ ]:
# SmolLM2 backend=vllm — one 800-item run
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "backend")


In [ ]:
# Qwen-3B backend=vllm — one 800-item run
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "backend")


In [ ]:
# Phi-3.5 backend=vllm — one 800-item run
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "backend")


Unpack the Drive zip on Mac. Commit new `results/runs/*.json` and `results/registry_paper.jsonl`. Do not edit numbers.
